# RDD Partitions, Partitioners, and Parallelism Inspection

**Module**: 02_rdd  
**Notebook**: 02_rdd_partitions.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: How RDDs are sliced into partitions, inspecting partition distribution with glom(), and controlling parallelism.
- **Why does this topic exist?**: Partitions are the atomic unit of parallelism in Spark. Misconfigured partitions lead to either idle cores or memory thrashing.
- **What problem does it solve?**: Under-partitioning wastes cluster resources; over-partitioning causes scheduling bottleneck.
- **Where is it used in real-world Data Engineering?**: Optimizing batch ETL runtimes and tuning data ingestion from external file systems.

---

## 2. Partition Mechanics

```
RDD (Distributed Collection)
+---------------------------------------------------------------+
|  Partition 0  |  Partition 1  |  Partition 2  |  Partition 3  |
+---------------+---------------+---------------+---------------+
        |               |               |               |
        v               v               v               v
    Task 0          Task 1          Task 2          Task 3
 (Core 1, W1)    (Core 2, W1)    (Core 1, W2)    (Core 2, W2)
```
- Each partition is assigned to exactly **one Task** running on **one CPU Core**.
- Default partition count when reading files is determined by total input splits (default 128 MB blocks).

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("RDD_Partitions_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

## 3. Inspecting Partitions with `getNumPartitions()` and `glom()`

In [ ]:
sc = spark.sparkContext
numbers = list(range(1, 25))

# Create RDD with 4 explicit partitions
rdd_4 = sc.parallelize(numbers, 4)
print(f"Partition count: {rdd_4.getNumPartitions()}")

# glom() gathers all elements of each partition into a Python list
partition_contents = rdd_4.glom().collect()
for idx, p in enumerate(partition_contents):
    print(f"  Partition {idx} (size {len(p)}): {p}")

## 4. `mapPartitions` and `mapPartitionsWithIndex`

`mapPartitions` processes an entire partition iterator at once, enabling connection reuse and batch operations.

In [ ]:
def compute_partition_stats(index, iterator):
    items = list(iterator)
    if items:
        yield {
            "partition": index,
            "count": len(items),
            "min": min(items),
            "max": max(items),
            "sum": sum(items)
        }

stats = rdd_4.mapPartitionsWithIndex(compute_partition_stats).collect()
print("Partition Statistics:")
for s in stats:
    print(" ", s)

---

## Experiment: Execution Latency vs Partition Count

- **1. What are we testing?**: Benchmark how execution duration varies when running on 1 partition vs 4 partitions on a 4-core cluster.
- **2. What do we expect?**: 1 partition utilizes only 1 core (sequential); 4 partitions utilize all 4 cores concurrently (~4x speedup).

In [ ]:
import math

def heavy_computation(x):
    # CPU-bound calculation
    return sum(math.sqrt(i) for i in range(1, 10000))

data = list(range(200))

# 1 Partition (Sequential)
t0 = time.time()
sc.parallelize(data, 1).map(heavy_computation).count()
t_1 = time.time() - t0

# 4 Partitions (Distributed across all 4 worker cores)
t0 = time.time()
sc.parallelize(data, 4).map(heavy_computation).count()
t_4 = time.time() - t0

print(f"1 Partition duration:  {t_1:.3f}s (1 core active, 3 idle)")
print(f"4 Partitions duration: {t_4:.3f}s (all 4 cores active)")
print(f"Speedup factor:        {t_1 / t_4:.2f}x")

**3. What happened?**: 4 partitions ran nearly 3-4x faster than 1 partition.  
**4. Why did it happen?**: Because work was distributed in parallel across all 4 available executor cores in our 2-worker cluster.  
**5. Practical Engineering Lesson**: Always ensure partition count is at least equal to or a multiple of total executor cores.

---

## Practical Exercises

### Exercise 1 (Beginner): Check Partitions of Retail Orders RDD
**Problem Statement**: Load /opt/spark-data/retail/orders.csv via textFile and print its default partition count.  
**Input / Context**: sc.textFile('/opt/spark-data/retail/orders.csv')

In [ ]:
# Exercise 1: Orders partition count

### Exercise 2 (Intermediate): Find Largest and Smallest Partition via glom()
**Problem Statement**: Use glom() and map() to calculate the minimum and maximum partition sizes for orders_rdd.  
**Input / Context**: orders_rdd

In [ ]:
# Exercise 2: Min and max partition sizes

### Exercise 3 (Advanced): Custom Hash Partitioning with partitionBy()
**Problem Statement**: Create a PairRDD of (customer_id, order_id) and partition it into 4 partitions using HashPartitioner.  
**Input / Context**: orders_rdd mapped to (customer_id, order_id)

In [ ]:
# Exercise 3: partitionBy

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Check Partitions of Retail Orders RDD):
```python
orders_rdd = sc.textFile("/opt/spark-data/retail/orders.csv")
print("Default partitions for orders.csv:", orders_rdd.getNumPartitions())
```

#### Solution 2 (Find Largest and Smallest Partition via glom()):
```python
sizes = orders_rdd.glom().map(len).collect()
print("Partition sizes:", sizes)
print(f"Min partition size: {min(sizes)}, Max partition size: {max(sizes)}")
```

#### Solution 3 (Custom Hash Partitioning with partitionBy()):
```python
header = orders_rdd.first()
kv_rdd = orders_rdd.filter(lambda r: r != header).map(lambda r: r.split(",")).map(lambda c: (int(c[1]), int(c[0])))
partitioned_rdd = kv_rdd.partitionBy(4)
print("New partition count:", partitioned_rdd.getNumPartitions())
print("Distribution per partition:", partitioned_rdd.glom().map(len).collect())
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Partitions = Parallelism**: 1 task processes 1 partition. Max concurrency = total available executor cores.
- **glom() Utility**: Useful for diagnostics; gathers partition items into lists.
- **mapPartitions Efficiency**: Enables batch operations and connection reuse per partition.

### Essential Technical Terms
- `Partition`: Atomic split of distributed data.
- `glom()`: Collects elements of each partition into an array.
- `mapPartitions`: Operates on an entire partition via an iterator.

### Common Engineering Mistakes
- **Single Partition Data**: Processing large data in 1 partition serializes execution and causes OOM.
- **Calling glom().collect() on Huge Partitions**: Brings entire partition lists into Driver memory, causing OOM.

### Interview Questions on This Topic
- **Q: What is the relationship between partitions and tasks?**
  - *A*: There is a strict 1:1 mapping: each stage creates exactly one task for every partition in its input RDD.
- **Q: Why use mapPartitions instead of map?**
  - *A*: To avoid per-row initialization overhead (e.g. database connections, regex compilation, heavy parsers).

### What to Study Next
Open 02_rdd/04_rdd_dependencies_and_lineage.ipynb to explore RDD fault tolerance and stage boundaries.

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')